In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel("Multilingual Language Data.xlsx")
df.head()

In [ ]:
df['Hasil'].unique()

In [ ]:
df['Category'].unique()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, savgol_filter

time = df["Time"].to_numpy(dtype=float)
time = np.round(time, decimals=2)
signal_noisy = df[["Signal 1", "Signal 2"]].to_numpy(dtype=float)

#Normalization function
def minmax(x):
    x = np.asarray(x, dtype=float)
    denom = (np.nanmax(x) - np.nanmin(x))
    if denom == 0 or np.isnan(denom):
        return np.zeros_like(x)
    return (x - np.nanmin(x)) / denom

signal_norm_1 = minmax(df["Signal 1"])
signal_norm_2 = minmax(df["Signal 2"])
signal_norm = np.column_stack([signal_norm_1, signal_norm_2])

#Savitzky-Golay Smoothing
def make_odd(n):
    n = int(n)
    return n if n % 2 == 1 else n + 1
N = len(df)
window_length = make_odd(25)
window_length = min(window_length, N if N % 2 == 1 else N - 1)
if window_length < 5:
    raise ValueError("Data terlalu sedikit untuk Savitzky-Golay. Perbesar N atau kecilkan window.")

polyorder = 3
if polyorder >= window_length:
    polyorder = window_length - 2

signal_sg = savgol_filter(signal_norm, window_length, polyorder, axis=0)
df["smooth_ch1"] = signal_sg[:, 0]
df["smooth_ch2"] = signal_sg[:, 1]

#Segmentation
segment_duration = 3.0
df["Segment"] = np.floor(df["Time"].to_numpy(dtype=float) / segment_duration).astype(int)
unique_segments = np.sort(df["Segment"].unique())

df.head()

In [ ]:
from scipy.signal import find_peaks
 
time_fil = df["Time"].values
signal_fil = df[['Signal 1', 'Signal 2']].values


plt.figure(figsize=(12, 6))
plt.plot(df["Time"], df[["smooth_ch1", "smooth_ch2"]], linewidth=1)
time_intervals = np.arange(time_fil[0], time_fil[-1], 3)
for t in time_intervals:
    plt.axvspan(t, t + 3, color='black', alpha=0.1)  # Highlight window 
plt.xlabel("Time (s)")
plt.ylabel("Signal Value")
plt.title("Windowing Signal")
plt.legend()
plt.show()

In [ ]:
import matplotlib.cm as cm

time_fil = df["Time"].values
signal_fil = df["Signal 1"].values
valleys, _ = find_peaks(-signal_fil, prominence=np.std(signal_fil))
segment_duration = 3
df["Segment"] = (df["Time"] // segment_duration).astype(int)
unique_segments = df["Segment"].unique()
cmap = cm.get_cmap('tab20', len(unique_segments))

plt.figure(figsize=(14, 6))
for i, segment_id in enumerate(unique_segments[:60]):
    segment_data = df[df["Segment"] == segment_id]
    plt.plot(segment_data["Time"], segment_data[["smooth_ch1", "smooth_ch2"]], label=f"Segment {segment_id+1}", color=cmap(i))
plt.xlabel("Time (s)")
plt.ylabel("Signal Value")
plt.title("Segmented Signal Visualization with Valleys")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Dropout, BatchNormalization, Bidirectional, LSTM, GlobalAveragePooling1D, Dense, LeakyReLU)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Set seed
def set_seed(seed=42):
    import os, random
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
set_seed(42)

features = df[["Signal 1", "Signal 2"]].values
labels = df["Category"].astype('category').cat.codes.values

# === Scaling & Augmentasi Jitter ===
scaler = MinMaxScaler(feature_range=(-1, 1))
features = scaler.fit_transform(features)

def add_jitter(data, sigma=0.01):
    return data + np.random.normal(0, sigma, data.shape)

features_aug = np.vstack([features, add_jitter(features)])
labels_aug = np.concatenate([labels, labels])

# === Windowing ===
def create_segmented_sequences(data, labels, window_size, step):
    sequences, target_labels = [], []
    for i in range(0, len(data) - window_size + 1, step):
        sequences.append(data[i:i+window_size])
        target_labels.append(labels[i + window_size - 1])
    return np.array(sequences), np.array(target_labels)

window_size = 75
step = 3
X, y = create_segmented_sequences(features_aug, labels_aug, window_size, step)

if len(X.shape) == 2:
    X = X.reshape(-1, window_size, 1)

# === Train-test split & class weighting ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

model = Sequential([
    Conv1D(filters=256, kernel_size=5, activation='relu', padding='same', input_shape=(window_size, X.shape[2])),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    GlobalAveragePooling1D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(np.unique(labels)), activation='softmax')
])

model.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)

history = model.fit(X_train, y_train, epochs=500, batch_size=32,validation_data=(X_test, y_test),
    callbacks=[reduce_lr, early_stop],
    class_weight=class_weight_dict
)

# === Evaluate ===
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

In [ ]:
from sklearn.metrics import  classification_report, confusion_matrix

classes = df['Category'].unique()

# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Normalization confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)
cm_normalized = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]
cm_normalized = np.around(cm_normalized, 2)
target_names = [str(c) for c in classes]

# Print confusion matrix after normalization
print("Normalized confusion matrix:")
print(cm_normalized)

# Plot confusion matrix
plt.figure(figsize=(50,50), dpi=300)
plt.imshow(cm_normalized, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix', fontsize=18, weight='bold')
plt.colorbar()
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, fontsize=14, font='Malgun Gothic')
plt.yticks(tick_marks, classes, fontsize=14, font='Malgun Gothic')

for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        plt.text(j, i, format(cm_normalized[i, j], '.2f'),
                 ha="center", va="center",
                 color="white" if cm_normalized[i, j] > 0.5 else "black")

plt.ylabel('Actual Label', fontsize=18, weight='bold')
plt.xlabel('Predicted Label', fontsize=18, weight='bold')
plt.tight_layout()
plt.show()

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=target_names))

print("Test Accuracy : {:.2f}%".format(accuracy*100))

In [ ]:
val_loss, val_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {final_train_acc * 100:.2f}%")
print(f"Final Validation Accuracy: {final_val_acc * 100:.2f}%")

final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
print(f"\nFinal Training Loss: {final_train_loss * 100:.2f}%")
print(f"Final Validation Loss: {final_val_loss * 100:.2f}%")

loss_gap = abs(final_train_loss-final_val_loss)
acc_gap = abs(final_train_acc-final_val_acc)

if  loss_gap < 0.02:
    print(f'\nGood')
    print(f'\nGap Loss: {loss_gap*100:.2f}%')
    print(f'Gap Accuracy: {acc_gap*100:.2f}%')
elif 0.02 <= loss_gap <= 0.1:
    print(f'\nUnderfitting')
    print(f'\nGap Loss: {loss_gap*100:.2f}%')
    print(f'Gap Accuracy: {acc_gap*100:.2f}%')
else:
    print(f'\nOverfitting')
    print(f'\nGap Loss: {loss_gap*100:.2f}%')
    print(f'Gap Accuracy: {acc_gap*100:.2f}%')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred_classes, target_names=target_names, output_dict=True)

precision = [report[cls]['precision'] for cls in target_names]
recall    = [report[cls]['recall'] for cls in target_names]
f1_score  = [report[cls]['f1-score'] for cls in target_names]

x = np.arange(len(target_names))
width = 0.5

# Plot histogram
fig, (ax1, ax2, ax3) = plt.subplots(ncols=3, figsize=(12, 10))
ax1.bar(x, precision, width, label='Precision')
ax2.bar(x, recall, width, label='Recall', color='orange')
ax3.bar(x, f1_score, width, label='F1-score', color='green')

# Label
ax1.set_ylabel('Precision')
ax1.set_xlabel('Letter')
ax1.set_title('Precision Number')
ax1.set_xticks(x)
ax1.set_xticklabels(target_names, font='Malgun Gothic')
ax1.set_ylim(0, 1.15)

ax2.set_ylabel('Recall')
ax2.set_xlabel('Letter')
ax2.set_title('Recall Number')
ax2.set_xticks(x)
ax2.set_xticklabels(target_names, font='Malgun Gothic')
ax2.set_ylim(0, 1.15)

ax3.set_ylabel('F1 Score')
ax3.set_xlabel('Letter')
ax3.set_title('F1 Score Number')
ax3.set_xticks(x)
ax3.set_xticklabels(target_names, font='Malgun Gothic')
ax3.set_ylim(0, 1.15)

plt.show()


In [ ]:
step = 2
epochs = range(len(history.history['accuracy']))
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = np.array(history.history['loss'])
val_loss = np.array(history.history['val_loss'])

def smooth_curve(data, factor=0.9):
    smoothed = []
    last = data[0]
    for point in data:
        last = factor * last + (1 - factor) * point
        smoothed.append(last)
    return smoothed
loss_smooth = smooth_curve(history.history["loss"])
val_loss_smooth = smooth_curve(history.history["val_loss"])
acc_smooth = smooth_curve(history.history["accuracy"])
val_acc_smooth = smooth_curve(history.history["val_accuracy"])

fig, ax1 = plt.subplots(figsize=(6,4))

# 🎨 Soft colors
colors = {
    'train_loss': '#FFDAB9',     # peach
    'val_loss': '#F4A6A6',       # soft red
    'train_acc': '#AEC6CF',      # pastel blue
    'val_acc': '#B4E197'         # pastel green
}

# Sumbu kiri: Loss (dari loss)
ax1.plot(epochs, loss_smooth, '--', color=colors['train_loss'], label='Train Loss', markersize=5)
ax1.plot(epochs, val_loss_smooth, '-', color=colors['val_loss'], label='Validation Loss', markersize=5)
ax1.plot(epochs[::step], loss_smooth[::step], 's', color=colors['train_loss'], markersize=5)
ax1.plot(epochs[::step], val_loss_smooth[::step], 's', color=colors['val_loss'], markersize=5)
ax1.set_xlabel('Epochs', weight='bold', fontsize=12)
ax1.set_ylabel('Loss', color=colors['val_loss'], weight='bold',fontsize=12)
ax1.tick_params(axis='y', labelcolor=colors['val_loss'])
ax1.set_ylim(-0.05, 1.1)
ax1.set_xticks(np.arange(0, len(epochs)+1, step))

# Sumbu kanan: Accuracy
ax2 = ax1.twinx()
ax2.plot(epochs, acc_smooth, '--', color=colors['train_acc'], label='Train Accuracy', markersize=5)
ax2.plot(epochs, val_acc_smooth, '-', color=colors['val_acc'], label='Validation Accuracy', markersize=5)
ax2.plot(epochs[::step], acc_smooth[::step], 'o', color=colors['train_acc'], markersize=5)
ax2.plot(epochs[::step], val_acc_smooth[::step], 'o', color=colors['val_acc'], markersize=5)
ax2.set_ylabel('Accuracy', color=colors['val_acc'], weight='bold',fontsize=12)
ax2.tick_params(axis='y', labelcolor=colors['val_acc'])
ax2.set_ylim(-0.05, 1.05)
ax1.set_xticks(np.arange(0, len(epochs)+1, step))

# Gabung legend dari dua axis
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='center right', frameon=True, facecolor='Floralwhite', ncol=2, fontsize=8)

# Final sentuhan
plt.title("Training & Validation Loss and Accuracy", weight = 'bold',fontsize=14)
# plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, Dropout, BatchNormalization,
    GlobalAveragePooling1D, Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# ===== Set seed =====
def set_seed(seed=42):
    import os, random
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

features = df[["Signal 1", "Signal 2"]].values
labels = df["category"].astype('category').cat.codes.values
n_classes = len(np.unique(labels))

scaler = MinMaxScaler(feature_range=(-1, 1))
features = scaler.fit_transform(features)
def add_jitter(data, sigma=0.01):
    return data + np.random.normal(0, sigma, data.shape)

features_aug = np.vstack([features, add_jitter(features)])
labels_aug = np.concatenate([labels, labels])

def create_segmented_sequences(data, labels, window_size, step):
    sequences, target_labels = [], []
    for i in range(0, len(data) - window_size + 1, step):
        sequences.append(data[i:i + window_size])
        target_labels.append(labels[i + window_size - 1])
    return np.array(sequences), np.array(target_labels)

window_size = 75
step = 3
X, y = create_segmented_sequences(features_aug, labels_aug, window_size, step)

if len(X.shape) == 2:
    X = X.reshape(-1, window_size, 1)

input_shape = (window_size, X.shape[2])

def build_cnn_model(input_shape, n_classes):
    model = Sequential([
        Conv1D(filters=256, kernel_size=5, activation='relu',
               padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        GlobalAveragePooling1D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5,
    min_lr=1e-6, verbose=1
)
early_stop = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=1
)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_no = 1
val_acc_scores = []
val_loss_scores = []

for train_idx, val_idx in kfold.split(X, y):
    print(f"\n===== Fold {fold_no} / 5 =====")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weight_dict = dict(enumerate(class_weights))

    model = build_cnn_model(input_shape, n_classes)

    history = model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[reduce_lr, early_stop],
        class_weight=class_weight_dict,
        verbose=1
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold_no} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc * 100:.2f}%")

    val_acc_scores.append(val_acc)
    val_loss_scores.append(val_loss)

    fold_no += 1

print("\n===== 5-Fold CV Result =====")
print(f"Val Accuracy per fold: {[f'{a*100:.2f}%' for a in val_acc_scores]}")
print(f"Mean Val Accuracy: {np.mean(val_acc_scores)*100:.2f}% ± {np.std(val_acc_scores)*100:.2f}%")
print(f"Mean Val Loss: {np.mean(val_loss_scores):.4f} ± {np.std(val_loss_scores):.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

folds = np.array([1, 2, 3, 4, 5])
accuracies = np.array(np.round(accs, 2))

# Calculate mean and standard deviation
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

# Create figure (transparent background, high resolution)
plt.figure(figsize=(5, 4), dpi=300, facecolor='none')
bars = plt.bar(
    folds, accuracies, color='#6BAED6', edgecolor='#08306B', width=0.6
)
plt.axhline(mean_acc, color='#08306B', linestyle='--', linewidth=1.2)
for i, acc in enumerate(accuracies):
    plt.text(folds[i], acc + 0.02, f"{acc*100:.2f} %",
             ha='center', va='bottom', fontsize=6, color='#08306B', fontweight='bold')
plt.ylim(min(accuracies) - 0.5, max(accuracies)+0.1)
plt.yticks(np.arange(0,0.78, 0.2))
plt.xticks(folds, [f"Fold {i}" for i in folds])
plt.ylabel("Validation Accuracy (%)", fontsize=10, color='#08306B', fontweight='bold')
plt.xlabel("Folds", fontsize=10, color='#08306B', fontweight='bold')
plt.title("Five-Fold Cross-Validation Accuracy", fontsize=10,color='#08306B', pad=10, fontweight='bold')
plt.text(1, min(accuracies) - 0.5, f"Mean ± s.d. = {mean_acc*100:.2f} ± {std_acc*100:.2f}%", fontsize=12, fontweight='bold', color='#08306B', 
         bbox=dict(facecolor='white', edgecolor='#08306B', 
                   boxstyle='round,pad=0.3', alpha=1), va='center')
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from tensorflow.keras.models import Sequential
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from matplotlib.lines import Line2D

X = data[['Signal 1', 'Signal 2']].to_numpy() 
y = data['Category'].to_numpy()
classes = data['Category'].unique()

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)
kmeans = KMeans(n_clusters=26, random_state=42, n_init=26)
clusters = kmeans.fit_predict(X_tsne)

cluster_map = {
    0: 'ENG',
    1: 'KOR',
    2: 'JPN'
}

plt.figure(figsize=(10.72, 8.2), dpi=300)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=clusters, cmap='tab20', s=10, alpha=0.8)

legend_elements = [
    Line2D([0], [0], marker='o',
        color='none',
        label=cluster_map[i],
        markerfacecolor=plt.cm.tab20(i),
        markersize=8
    )
    for i in sorted(cluster_map.keys())
]

leg = plt.legend(handles=legend_elements, title='Language',
                 bbox_to_anchor=(1.02, 1), loc='upper left',
                 frameon=False, fontsize=18, title_fontsize=20)
if leg is not None: 
    leg.get_title().set_fontweight('bold')

plt.title('t-SNE Visualization', fontsize=28, weight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=28, weight='bold')
plt.ylabel('t-SNE Dimension 2', fontsize=28, weight='bold')
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
#Comparison model

x = df[['Signal 1', 'Signal 2']]
y = df['Category']
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)
print(f"x_train : ({x_train.shape})")
print(f"x_test : ({x_test.shape})")
print(f"y_train : ({y_train.shape})")
print(f"y_test : ({y_test.shape})")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

imputer = SimpleImputer(strategy='mean')
x_train = imputer.fit_transform(x_train)
x_test = imputer.transform(x_test)

# KNN Classification
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train, y_train)
y_pred_knn = knn.predict(x_test)
KNN_accuracy = accuracy_score(y_test, y_pred_knn)
print(classification_report(y_test, y_pred_knn,zero_division=1))
print("KNN Accuracy: {:.2f}%".format(KNN_accuracy * 100))

# Decision Tree Classification
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)
y_pred_dt = dt.predict(x_test)
DT_accuracy = accuracy_score(y_test, y_pred_dt)
print(classification_report(y_test, y_pred_dt,zero_division=1))
print("Decision Tree Accuracy: {:.2f}%".format(DT_accuracy * 100))

# Random Forest Classification
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)
RF_accuracy = accuracy_score(y_test, y_pred_rf)
print(classification_report(y_test, y_pred_rf,zero_division=1))
print("Random Forest Accuracy: {:.2f}%".format(RF_accuracy * 100))


# Add other models like KNN, Decision Tree, etc. and compare
models_accuracies = {
    "KNN": KNN_accuracy,
    "Random Forest": RF_accuracy,
    "Decision Tree": DT_accuracy

}

BOLD = '\033[1m'
RED = '\033[31m'   # Red color
GREEN = '\033[32m' # Green color
BLUE = '\033[34m'  # Blue color
END = '\033[0m'    # Reset formatting

# Find the model with the best accuracy
best_model_name = max(models_accuracies, key=models_accuracies.get)
print(f"Best Model: {BOLD}{BLUE}{best_model_name}{END} with Accuracy: {models_accuracies[best_model_name] * 100:.2f}%")
models_accuracies = {
    "KNN": KNN_accuracy,
    "Random Forest": RF_accuracy,
    "Decision Tree": DT_accuracy
}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Model predictions
y_pred_knn = knn.predict(x_test)
y_pred_dt = dt.predict(x_test)
y_pred_rf = rf.predict(x_test)

# Confusion matrices
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)

# Normalization function
def normalize_cm(cm):
    return np.around(cm.astype('float') / cm.sum(axis=1)[:, np.newaxis], 2)

cm_normalized_knn = normalize_cm(cm_knn)
cm_normalized_dt = normalize_cm(cm_dt)
cm_normalized_rf = normalize_cm(cm_rf)

# Define class labels
classes = df['Category'].unique()

# Define function to plot confusion matrix
def plot_confusion_matrix(cm, title,cmap='Reds'):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap=cmap, xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted', weight='bold', fontsize=18)
    plt.ylabel('Actual', weight='bold', fontsize=18)
    plt.yticks(rotation=360, fontsize=14)
    plt.xticks(fontsize=14)
    plt.title(title,fontsize=18, weight='bold')
    plt.show()

# Plot all confusion matrices
plot_confusion_matrix(cm_normalized_knn, "KNN Confusion Matrix", cmap='Reds')
plot_confusion_matrix(cm_normalized_dt, "Decision Tree Confusion Matrix", cmap='BuGn')
plot_confusion_matrix(cm_normalized_rf, "Random Forest Confusion Matrix", cmap='RdPu')
